# Week 4: Application, SQL, API and reproducible handover

This notebook uses the supplied PDF’s Python data-science workflow. Run cells in order. Model fitting and evaluation are real computations; the bundled source is clearly labelled synthetic practice data.

## Learn, practise, complete
**Topics:** Python Streamlit, Plotly, FastAPI schemas, SQLite/SQL, persistence, tests, reproducible handover.

**YouTube:** [Data Professor Streamlit](https://www.youtube.com/@DataProfessor/search?query=streamlit), [freeCodeCamp SQL](https://www.youtube.com/@freecodecamp/search?query=sql%20beginners).

**Websites:** [Streamlit](https://docs.streamlit.io/get-started), [FastAPI](https://fastapi.tiangolo.com/tutorial/), [SQLBolt](https://sqlbolt.com/), [pytest](https://docs.pytest.org/en/stable/getting-started.html).

**Your task:** demonstrate the dashboard, export forecasts and assumptions, test malformed inputs, prepare your report and slides, and explain starter code versus your own work. Keep meaningful daily Git activity as required by the execution PDF.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT/'src/data.py').is_file(), 'Open this notebook from the project or notebooks folder.'
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib','inline')
from src.data import read_sales, source_info, file_hash

# Change this to a licensed canonical sales CSV for your own analysis.
DATA_PATH = ROOT/'data/raw/demo_sales.csv'
data = read_sales(DATA_PATH)
source = source_info(DATA_PATH)
print('DATA SOURCE:', source['data_kind'])
print('SOURCE SHA-256:', source['source_sha256'])
print('Bundled data is synthetic practice data, not real retail observations.')
display(data.head())

DATA SOURCE: synthetic practice; not observed retail sales
SOURCE SHA-256: c0f8a91575ec1819cb02c4783d23f8e7d3d172d1cd83fce572aaff57302c67fc
Bundled data is synthetic practice data, not real retail observations.


,date,store,item,sales
0,2022-01-01,1,1,32
1,2022-01-02,1,1,28
2,2022-01-03,1,1,18
3,2022-01-04,1,1,11
4,2022-01-05,1,1,16


## 1. Reuse a matching saved run without fitting on startup

In [2]:
from src.artifacts import restore_run
MODEL_DIR=ROOT/'models/demo'
REPORT_DIR=ROOT/'reports/evaluation/demo'
if DATA_PATH.name!='demo_sales.csv':
    MODEL_DIR=ROOT/'models/retail'; REPORT_DIR=ROOT/'reports/evaluation/retail'
result=restore_run(data,MODEL_DIR,REPORT_DIR)
assert result is not None,'Run Week 3 for the chosen data first.'
print('Loaded:',result['manifest']['selected_model'])
display(result['forecasts'].head())

Loaded: random_forest


,date,store,item,prediction,model
0,2024-01-01,1,1,23.058641,random_forest
1,2024-01-01,1,2,27.851361,random_forest
2,2024-01-01,1,3,29.985103,random_forest
3,2024-01-01,1,4,38.954726,random_forest
4,2024-01-01,2,1,22.582587,random_forest


## 2. Query the actual forecast and evaluation outputs with SQL

In [3]:
from src.reporting import export_database
import sqlite3
sql_result=export_database(REPORT_DIR)
with sqlite3.connect(REPORT_DIR/'results.sqlite') as connection:
    summary=pd.read_sql_query('SELECT model, AVG(ABS(prediction-actual)) AS mae, COUNT(*) AS observations FROM test_predictions GROUP BY model ORDER BY mae',connection)
    volume=pd.read_sql_query('SELECT store,item,SUM(prediction) AS next_seven_day_units FROM forecasts GROUP BY store,item',connection)
display(summary); display(volume)
print('SQL reports existing results after model selection; it does not select the model.')

,model,mae,observations
0,linear_regression,5.215728,224
1,random_forest,5.342322,224
2,seasonal_naive,6.455357,224
3,naive,10.312500,224


,store,item,next_seven_day_units
0,1,1,200.525287
1,1,2,247.005759
2,1,3,270.822646
3,1,4,330.595529
4,2,1,216.370263
5,2,2,267.033996
6,2,3,315.437215
7,2,4,354.605293


SQL reports existing results after model selection; it does not select the model.


## 3. Exercise the FastAPI Python service locally

In [4]:
import os
os.environ['DEMAND_SALES_CSV']=str(DATA_PATH)
os.environ['DEMAND_MODEL_PATH']=str(MODEL_DIR/'selected_model.joblib')
from fastapi.testclient import TestClient
from api.main import app
client=TestClient(app)
assert client.get('/health').status_code==200
catalog=client.get('/models').json()
display(pd.DataFrame(catalog['models'])[['name','file','selected','type']])
key=next(iter(data.groupby(['store','item']).groups))
response=client.post('/forecast',json={'store':key[0],'item':key[1],'horizon':7,'method':'selected'})
response.raise_for_status()
api_forecast=pd.DataFrame(response.json()['forecasts'])
display(api_forecast)
assert len(api_forecast)==7
assert client.post('/forecast',json={'store':key[0],'item':key[1],'horizon':99}).status_code==422

,name,file,selected,type
0,naive,naive.json,False,baseline specification
1,seasonal_naive,seasonal_naive.json,False,baseline specification
2,linear_regression,linear_regression.joblib,False,fitted scikit-learn pipeline
3,random_forest,random_forest.joblib,True,fitted scikit-learn pipeline


,date,store,item,prediction,model
0,2024-01-01,1,1,23.058641,random_forest
1,2024-01-02,1,1,23.136915,random_forest
2,2024-01-03,1,1,25.567512,random_forest
3,2024-01-04,1,1,31.334709,random_forest
4,2024-01-05,1,1,30.530946,random_forest
5,2024-01-06,1,1,35.804553,random_forest
6,2024-01-07,1,1,31.092010,random_forest


## 4. Verify the API and saved model give the same forecast

In [5]:
expected=result['forecasts']
expected=expected[(expected.store==key[0]) & (expected.item==key[1])]
np.testing.assert_allclose(api_forecast.prediction,expected.prediction,rtol=1e-10,atol=1e-10)
print('PASS: API and saved-model forecasts match.')

PASS: API and saved-model forecasts match.


## 5. Start the dashboard or API from a terminal
Do not launch a blocking server inside this Run All workflow. Open a project terminal with the same virtual environment and run one of these Python commands:

```text
python start_dashboard.py
python -m streamlit run app/dashboard.py
python -m uvicorn api.main:app --host 127.0.0.1 --port 8000
```

The dashboard has Sales overview, Forecasts, Inventory scenarios, Model performance and Models tabs. Matching saved models load after restarting; no mandatory retraining is required. New dashboard training saves to `models/runs/<dataset-id>/` and matching reports under `reports/runs/`.

## 6. Inspect package versions and the model card

In [6]:
from importlib.metadata import version
import json
display(pd.Series({name:version(name) for name in ['numpy','pandas','scikit-learn','statsmodels','streamlit','plotly','fastapi','jupyterlab']}).to_frame('installed version'))
display(pd.Series(json.loads((MODEL_DIR/'model_card.json').read_text())).to_frame('model card'))
print('Run from the terminal for the complete suite: python -m pytest -q')

,installed version
numpy,2.3.5
pandas,2.2.3
scikit-learn,1.8.0
statsmodels,0.14.5
streamlit,1.50.0
plotly,6.3.1
fastapi,0.115.12
jupyterlab,4.4.9


,model card
forecast_horizon_days,7
independent_business_validation,False
source,{'data_kind': 'synthetic practice; not observe...
selected_model,random_forest
selection,Lowest validation MAE; test does not select th...
training_end,2023-12-31
canonical_data_sha256,c0f8a91575ec1819cb02c4783d23f8e7d3d172d1cd83fc...
limitations,[Synthetic practice models are not real retail...
packages,"{'numpy': '2.3.5', 'pandas': '2.2.3', 'scikit-..."
trusted_load,Only deserialize artifacts generated by your o...


Run from the terminal for the complete suite: python -m pytest -q


## Your final submission
Use `docs/FINAL_REPORT_TEMPLATE.md`: business problem, data terms/source, cleaning/exclusions, EDA, chronological evaluation, baseline comparison, inventory assumptions, application screenshots, tests, limitations and contributions. Keep meaningful daily Git activity; weekly tags alone do not meet the execution PDF’s commit-day rules.

GitHub’s language percentages are not prescribed by the PDF. This project’s implementation languages are Python and Python Jupyter notebooks; SQL exercises run through Python. Do not change language statistics to imitate another repository.